# Model Development for Citation Impact Prediction

This notebook trains and evaluates ML models for:
- **Classification**: Predicting HighImpact papers (top 25%)
- **Regression**: Predicting Citations_log

**Models:**
- Baseline: Logistic Regression, Linear Regression
- Tree-based: Random Forest, XGBoost, LightGBM

**Evaluation Strategy:**
- Random split (80/20): Best-case performance
- Temporal split (<2021 vs 2021+): Real-world scenario

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
from datetime import datetime

# Scikit-learn models
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# XGBoost and LightGBM
import xgboost as xgb
import lightgbm as lgb

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score
)

# Set random seed
np.random.seed(42)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully!")

## 1. Load Processed Data

In [ ]:
# Load feature metadata
with open('../models/feature_info.json', 'r') as f:
    feature_info = json.load(f)

print("Feature Information:")
print(f"  Total features: {feature_info['total_features']}")
print(f"  Base features: {feature_info['n_base']}")
print(f"  Collaboration features: {feature_info['n_collaboration']}")
print(f"  TF-IDF features: {feature_info['n_tfidf']}")
print(f"\n  Temporal cutoff: {feature_info['temporal_cutoff']}")
print(f"  Train size (temporal): {feature_info['train_size_temporal']}")
print(f"  Test size (temporal): {feature_info['test_size_temporal']}")

In [ ]:
# Load random split data
print("Loading random split data...")
X_train_random = pd.read_csv('../data/X_train_random.csv')
X_test_random = pd.read_csv('../data/X_test_random.csv')
y_reg_train_random = pd.read_csv('../data/y_reg_train_random.csv')['Citations_log']
y_reg_test_random = pd.read_csv('../data/y_reg_test_random.csv')['Citations_log']
y_cls_train_random = pd.read_csv('../data/y_cls_train_random.csv')['HighImpact']
y_cls_test_random = pd.read_csv('../data/y_cls_test_random.csv')['HighImpact']

print(f"✓ Random split loaded")
print(f"  Train: {X_train_random.shape}, Test: {X_test_random.shape}")

In [ ]:
# Load temporal split data
print("Loading temporal split data...")
X_train_temporal = pd.read_csv('../data/X_train_temporal.csv')
X_test_temporal = pd.read_csv('../data/X_test_temporal.csv')
y_reg_train_temporal = pd.read_csv('../data/y_reg_train_temporal.csv')['Citations_log']
y_reg_test_temporal = pd.read_csv('../data/y_reg_test_temporal.csv')['Citations_log']
y_cls_train_temporal = pd.read_csv('../data/y_cls_train_temporal.csv')['HighImpact']
y_cls_test_temporal = pd.read_csv('../data/y_cls_test_temporal.csv')['HighImpact']

print(f"✓ Temporal split loaded")
print(f"  Train: {X_train_temporal.shape}, Test: {X_test_temporal.shape}")
print(f"\n  Train class balance: {y_cls_train_temporal.value_counts(normalize=True).to_dict()}")
print(f"  Test class balance: {y_cls_test_temporal.value_counts(normalize=True).to_dict()}")

## 2. Helper Functions for Evaluation

In [ ]:
def evaluate_classification(y_true, y_pred, y_proba=None, model_name="Model"):
    """
    Evaluate classification model and return metrics
    """
    results = {
        'model': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1_score': f1_score(y_true, y_pred)
    }
    
    if y_proba is not None:
        results['auc_roc'] = roc_auc_score(y_true, y_proba)
    
    return results

def evaluate_regression(y_true, y_pred, model_name="Model"):
    """
    Evaluate regression model and return metrics
    """
    results = {
        'model': model_name,
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae': mean_absolute_error(y_true, y_pred),
        'r2': r2_score(y_true, y_pred)
    }
    
    return results

def print_classification_report(results):
    """
    Pretty print classification results
    """
    print(f"\n{results['model']}:")
    print(f"  Accuracy:  {results['accuracy']:.4f}")
    print(f"  Precision: {results['precision']:.4f}")
    print(f"  Recall:    {results['recall']:.4f}")
    print(f"  F1-Score:  {results['f1_score']:.4f}")
    if 'auc_roc' in results:
        print(f"  AUC-ROC:   {results['auc_roc']:.4f}")

def print_regression_report(results):
    """
    Pretty print regression results
    """
    print(f"\n{results['model']}:")
    print(f"  RMSE: {results['rmse']:.4f}")
    print(f"  MAE:  {results['mae']:.4f}")
    print(f"  R²:   {results['r2']:.4f}")

print("✓ Helper functions defined")

## 3. Classification Models

Predicting **HighImpact** (top 25% of papers by citations)

### 3.1 Baseline: Logistic Regression

In [ ]:
print("="*80)
print("CLASSIFICATION: Logistic Regression (Baseline)")
print("="*80)

# Train on random split
lr_random = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_random.fit(X_train_random, y_cls_train_random)

# Predictions
y_pred_lr_random = lr_random.predict(X_test_random)
y_proba_lr_random = lr_random.predict_proba(X_test_random)[:, 1]

# Evaluate
lr_random_results = evaluate_classification(
    y_cls_test_random, y_pred_lr_random, y_proba_lr_random,
    model_name="Logistic Regression (Random Split)"
)
print_classification_report(lr_random_results)

# Train on temporal split
lr_temporal = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr_temporal.fit(X_train_temporal, y_cls_train_temporal)

# Predictions
y_pred_lr_temporal = lr_temporal.predict(X_test_temporal)
y_proba_lr_temporal = lr_temporal.predict_proba(X_test_temporal)[:, 1]

# Evaluate
lr_temporal_results = evaluate_classification(
    y_cls_test_temporal, y_pred_lr_temporal, y_proba_lr_temporal,
    model_name="Logistic Regression (Temporal Split)"
)
print_classification_report(lr_temporal_results)

### 3.2 Random Forest Classifier

In [ ]:
print("="*80)
print("CLASSIFICATION: Random Forest")
print("="*80)

# Train on random split
rf_random = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
rf_random.fit(X_train_random, y_cls_train_random)

# Predictions
y_pred_rf_random = rf_random.predict(X_test_random)
y_proba_rf_random = rf_random.predict_proba(X_test_random)[:, 1]

# Evaluate
rf_random_results = evaluate_classification(
    y_cls_test_random, y_pred_rf_random, y_proba_rf_random,
    model_name="Random Forest (Random Split)"
)
print_classification_report(rf_random_results)

# Train on temporal split
rf_temporal = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
rf_temporal.fit(X_train_temporal, y_cls_train_temporal)

# Predictions
y_pred_rf_temporal = rf_temporal.predict(X_test_temporal)
y_proba_rf_temporal = rf_temporal.predict_proba(X_test_temporal)[:, 1]

# Evaluate
rf_temporal_results = evaluate_classification(
    y_cls_test_temporal, y_pred_rf_temporal, y_proba_rf_temporal,
    model_name="Random Forest (Temporal Split)"
)
print_classification_report(rf_temporal_results)

### 3.3 XGBoost Classifier

In [ ]:
print("="*80)
print("CLASSIFICATION: XGBoost")
print("="*80)

# Calculate scale_pos_weight for class imbalance
scale_pos_weight_random = (y_cls_train_random == 0).sum() / (y_cls_train_random == 1).sum()
scale_pos_weight_temporal = (y_cls_train_temporal == 0).sum() / (y_cls_train_temporal == 1).sum()

# Train on random split
xgb_random = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight_random,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)
xgb_random.fit(X_train_random, y_cls_train_random)

# Predictions
y_pred_xgb_random = xgb_random.predict(X_test_random)
y_proba_xgb_random = xgb_random.predict_proba(X_test_random)[:, 1]

# Evaluate
xgb_random_results = evaluate_classification(
    y_cls_test_random, y_pred_xgb_random, y_proba_xgb_random,
    model_name="XGBoost (Random Split)"
)
print_classification_report(xgb_random_results)

# Train on temporal split
xgb_temporal = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight_temporal,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)
xgb_temporal.fit(X_train_temporal, y_cls_train_temporal)

# Predictions
y_pred_xgb_temporal = xgb_temporal.predict(X_test_temporal)
y_proba_xgb_temporal = xgb_temporal.predict_proba(X_test_temporal)[:, 1]

# Evaluate
xgb_temporal_results = evaluate_classification(
    y_cls_test_temporal, y_pred_xgb_temporal, y_proba_xgb_temporal,
    model_name="XGBoost (Temporal Split)"
)
print_classification_report(xgb_temporal_results)

### 3.4 LightGBM Classifier

In [ ]:
print("="*80)
print("CLASSIFICATION: LightGBM")
print("="*80)

# Train on random split
lgb_random = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight_random,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_random.fit(X_train_random, y_cls_train_random)

# Predictions
y_pred_lgb_random = lgb_random.predict(X_test_random)
y_proba_lgb_random = lgb_random.predict_proba(X_test_random)[:, 1]

# Evaluate
lgb_random_results = evaluate_classification(
    y_cls_test_random, y_pred_lgb_random, y_proba_lgb_random,
    model_name="LightGBM (Random Split)"
)
print_classification_report(lgb_random_results)

# Train on temporal split
lgb_temporal = lgb.LGBMClassifier(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight_temporal,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_temporal.fit(X_train_temporal, y_cls_train_temporal)

# Predictions
y_pred_lgb_temporal = lgb_temporal.predict(X_test_temporal)
y_proba_lgb_temporal = lgb_temporal.predict_proba(X_test_temporal)[:, 1]

# Evaluate
lgb_temporal_results = evaluate_classification(
    y_cls_test_temporal, y_pred_lgb_temporal, y_proba_lgb_temporal,
    model_name="LightGBM (Temporal Split)"
)
print_classification_report(lgb_temporal_results)

### 3.5 Classification Results Summary

In [ ]:
# Compile all classification results
classification_results = pd.DataFrame([
    lr_random_results,
    lr_temporal_results,
    rf_random_results,
    rf_temporal_results,
    xgb_random_results,
    xgb_temporal_results,
    lgb_random_results,
    lgb_temporal_results
])

print("="*80)
print("CLASSIFICATION RESULTS SUMMARY")
print("="*80)
print(classification_results.to_string(index=False))

# Save to CSV
classification_results.to_csv('../results/classification_results.csv', index=False)
print("\n✓ Results saved to results/classification_results.csv")

## 4. Regression Models

Predicting **Citations_log** (log-transformed citation counts)

### 4.1 Baseline: Linear Regression

In [ ]:
print("="*80)
print("REGRESSION: Linear Regression (Baseline)")
print("="*80)

# Train on random split
linreg_random = LinearRegression(n_jobs=-1)
linreg_random.fit(X_train_random, y_reg_train_random)

# Predictions
y_pred_linreg_random = linreg_random.predict(X_test_random)

# Evaluate
linreg_random_results = evaluate_regression(
    y_reg_test_random, y_pred_linreg_random,
    model_name="Linear Regression (Random Split)"
)
print_regression_report(linreg_random_results)

# Train on temporal split
linreg_temporal = LinearRegression(n_jobs=-1)
linreg_temporal.fit(X_train_temporal, y_reg_train_temporal)

# Predictions
y_pred_linreg_temporal = linreg_temporal.predict(X_test_temporal)

# Evaluate
linreg_temporal_results = evaluate_regression(
    y_reg_test_temporal, y_pred_linreg_temporal,
    model_name="Linear Regression (Temporal Split)"
)
print_regression_report(linreg_temporal_results)

### 4.2 Random Forest Regressor

In [ ]:
print("="*80)
print("REGRESSION: Random Forest")
print("="*80)

# Train on random split
rfr_random = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
rfr_random.fit(X_train_random, y_reg_train_random)

# Predictions
y_pred_rfr_random = rfr_random.predict(X_test_random)

# Evaluate
rfr_random_results = evaluate_regression(
    y_reg_test_random, y_pred_rfr_random,
    model_name="Random Forest (Random Split)"
)
print_regression_report(rfr_random_results)

# Train on temporal split
rfr_temporal = RandomForestRegressor(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    verbose=1
)
rfr_temporal.fit(X_train_temporal, y_reg_train_temporal)

# Predictions
y_pred_rfr_temporal = rfr_temporal.predict(X_test_temporal)

# Evaluate
rfr_temporal_results = evaluate_regression(
    y_reg_test_temporal, y_pred_rfr_temporal,
    model_name="Random Forest (Temporal Split)"
)
print_regression_report(rfr_temporal_results)

### 4.3 XGBoost Regressor

In [ ]:
print("="*80)
print("REGRESSION: XGBoost")
print("="*80)

# Train on random split
xgbr_random = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)
xgbr_random.fit(X_train_random, y_reg_train_random)

# Predictions
y_pred_xgbr_random = xgbr_random.predict(X_test_random)

# Evaluate
xgbr_random_results = evaluate_regression(
    y_reg_test_random, y_pred_xgbr_random,
    model_name="XGBoost (Random Split)"
)
print_regression_report(xgbr_random_results)

# Train on temporal split
xgbr_temporal = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)
xgbr_temporal.fit(X_train_temporal, y_reg_train_temporal)

# Predictions
y_pred_xgbr_temporal = xgbr_temporal.predict(X_test_temporal)

# Evaluate
xgbr_temporal_results = evaluate_regression(
    y_reg_test_temporal, y_pred_xgbr_temporal,
    model_name="XGBoost (Temporal Split)"
)
print_regression_report(xgbr_temporal_results)

### 4.4 LightGBM Regressor

In [ ]:
print("="*80)
print("REGRESSION: LightGBM")
print("="*80)

# Train on random split
lgbr_random = lgb.LGBMRegressor(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbr_random.fit(X_train_random, y_reg_train_random)

# Predictions
y_pred_lgbr_random = lgbr_random.predict(X_test_random)

# Evaluate
lgbr_random_results = evaluate_regression(
    y_reg_test_random, y_pred_lgbr_random,
    model_name="LightGBM (Random Split)"
)
print_regression_report(lgbr_random_results)

# Train on temporal split
lgbr_temporal = lgb.LGBMRegressor(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgbr_temporal.fit(X_train_temporal, y_reg_train_temporal)

# Predictions
y_pred_lgbr_temporal = lgbr_temporal.predict(X_test_temporal)

# Evaluate
lgbr_temporal_results = evaluate_regression(
    y_reg_test_temporal, y_pred_lgbr_temporal,
    model_name="LightGBM (Temporal Split)"
)
print_regression_report(lgbr_temporal_results)

### 4.5 Regression Results Summary

In [ ]:
# Compile all regression results
regression_results = pd.DataFrame([
    linreg_random_results,
    linreg_temporal_results,
    rfr_random_results,
    rfr_temporal_results,
    xgbr_random_results,
    xgbr_temporal_results,
    lgbr_random_results,
    lgbr_temporal_results
])

print("="*80)
print("REGRESSION RESULTS SUMMARY")
print("="*80)
print(regression_results.to_string(index=False))

# Save to CSV
regression_results.to_csv('../results/regression_results.csv', index=False)
print("\n✓ Results saved to results/regression_results.csv")

## 5. Feature Importance Analysis

In [ ]:
# Get feature importances from best tree-based models
# Using Random Forest on random split for interpretation

print("="*80)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80)

# Classification feature importance
feature_importance_cls = pd.DataFrame({
    'feature': X_train_random.columns,
    'importance': rf_random.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 Features for Classification (HighImpact):")
print(feature_importance_cls.head(20).to_string(index=False))

# Regression feature importance
feature_importance_reg = pd.DataFrame({
    'feature': X_train_random.columns,
    'importance': rfr_random.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 20 Features for Regression (Citations_log):")
print(feature_importance_reg.head(20).to_string(index=False))

In [ ]:
# Plot top 20 features for classification
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Classification
top20_cls = feature_importance_cls.head(20)
axes[0].barh(range(20), top20_cls['importance'].values)
axes[0].set_yticks(range(20))
axes[0].set_yticklabels(top20_cls['feature'].values)
axes[0].invert_yaxis()
axes[0].set_xlabel('Importance')
axes[0].set_title('Top 20 Features - Classification (HighImpact)')

# Regression
top20_reg = feature_importance_reg.head(20)
axes[1].barh(range(20), top20_reg['importance'].values)
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(top20_reg['feature'].values)
axes[1].invert_yaxis()
axes[1].set_xlabel('Importance')
axes[1].set_title('Top 20 Features - Regression (Citations_log)')

plt.tight_layout()
plt.savefig('../results/feature_importance.png', dpi=300, bbox_inches='tight')
print("\n✓ Feature importance plot saved to results/feature_importance.png")
plt.show()

In [ ]:
# Save feature importances to CSV
feature_importance_cls.to_csv('../results/feature_importance_classification.csv', index=False)
feature_importance_reg.to_csv('../results/feature_importance_regression.csv', index=False)
print("✓ Feature importance data saved to results/")

## 6. Save Best Models

In [ ]:
import os
os.makedirs('../models/saved', exist_ok=True)

print("Saving best models...")

# Save classification models (using random split for deployment)
with open('../models/saved/logistic_regression_classifier.pkl', 'wb') as f:
    pickle.dump(lr_random, f)

with open('../models/saved/random_forest_classifier.pkl', 'wb') as f:
    pickle.dump(rf_random, f)

with open('../models/saved/xgboost_classifier.pkl', 'wb') as f:
    pickle.dump(xgb_random, f)

with open('../models/saved/lightgbm_classifier.pkl', 'wb') as f:
    pickle.dump(lgb_random, f)

print("✓ Classification models saved")

# Save regression models
with open('../models/saved/linear_regression.pkl', 'wb') as f:
    pickle.dump(linreg_random, f)

with open('../models/saved/random_forest_regressor.pkl', 'wb') as f:
    pickle.dump(rfr_random, f)

with open('../models/saved/xgboost_regressor.pkl', 'wb') as f:
    pickle.dump(xgbr_random, f)

with open('../models/saved/lightgbm_regressor.pkl', 'wb') as f:
    pickle.dump(lgbr_random, f)

print("✓ Regression models saved")
print("\nAll models saved to models/saved/")

## 7. Summary and Conclusions

In [ ]:
print("="*80)
print("MODEL DEVELOPMENT COMPLETE")
print("="*80)

print("\n📊 MODELS TRAINED:")
print("  Classification (HighImpact):")
print("    ✓ Logistic Regression")
print("    ✓ Random Forest")
print("    ✓ XGBoost")
print("    ✓ LightGBM")
print("\n  Regression (Citations_log):")
print("    ✓ Linear Regression")
print("    ✓ Random Forest")
print("    ✓ XGBoost")
print("    ✓ LightGBM")

print("\n📂 OUTPUT FILES:")
print("  ✓ results/classification_results.csv")
print("  ✓ results/regression_results.csv")
print("  ✓ results/feature_importance_classification.csv")
print("  ✓ results/feature_importance_regression.csv")
print("  ✓ results/feature_importance.png")
print("  ✓ models/saved/ (8 trained models)")

print("\n🎯 BEST MODELS:")
print("  Classification (Random Split):")
best_cls = classification_results[classification_results['model'].str.contains('Random Split')].sort_values('f1_score', ascending=False).iloc[0]
print(f"    {best_cls['model']}: F1={best_cls['f1_score']:.4f}, AUC={best_cls['auc_roc']:.4f}")

print("\n  Regression (Random Split):")
best_reg = regression_results[regression_results['model'].str.contains('Random Split')].sort_values('r2', ascending=False).iloc[0]
print(f"    {best_reg['model']}: R²={best_reg['r2']:.4f}, RMSE={best_reg['rmse']:.4f}")

print("\n🚀 NEXT STEPS:")
print("  1. Hyperparameter tuning on best models")
print("  2. Deploy models to Streamlit app")
print("  3. Test real-time predictions")
print("  4. Create model insights dashboard")

print("\n" + "="*80)